In [ ]:
%py
# PySpark script to mask the last 4 digits of invoice_number in the purgo_playground.d_product_revenue_clone table

# Import necessary functions
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# Commented out SparkSession initialization, as spark is already available in Databricks
# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("Masking Invoice Number").getOrCreate()

# Load configurations
original_table = "purgo_playground.d_product_revenue"
clone_table = "purgo_playground.d_product_revenue_clone"

# Drop the clone table if it exists
spark.sql(f"DROP TABLE IF EXISTS {clone_table}")

# Create a replica of the original table
spark.sql(f"CREATE TABLE {clone_table} AS SELECT * FROM {original_table}")

# Load the clone table into a DataFrame
df_clone = spark.table(clone_table)

# Define masking function for invoice_number
def mask_invoice_number(invoice):
    invoice_str = str(invoice)
    if len(invoice_str) < 4:
        raise ValueError("Error: Invoice number less than expected length")
    return invoice_str[:-4] + '****'

# Register UDF for masking
mask_invoice_udf = F.udf(mask_invoice_number, StringType())

# Apply the masking function
df_masked = df_clone.withColumn("invoice_number", mask_invoice_udf("invoice_number"))

# Write the masked data back to the clone table with structure validation
if len(df_clone.columns) == len(df_masked.columns):
    df_masked.write.mode("overwrite").saveAsTable(clone_table)
else:
    raise ValueError("Error: Column count mismatch between original and masked data")

# Commented out stop command, as it can cause issues in Databricks
# spark.stop()